# Analyze protection/deprotection strategies

This notebook runs the route-level protection analysis added in `route_analysis.protection`. It detects deprotection events from the curated chython rules, traces the protected atom/group backward through a normalized PaRoutes route, and writes the aggregate protection output tables.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import SVG, display

PROJECT = Path('/Users/almazgil/Desktop/projects/Retro-BLEU')
ROUTES_JSON = PROJECT / 'PaRoutes/data/n1-routes.json'
COMPOSITE_RULE_DIR = PROJECT / 'composite_rules/comp_output/n1'
# OUTPUT_DIR = PROJECT / 'composite_rules/protection_out/n1_old'
OUTPUT_DIR = PROJECT / 'composite_rules/protection_out/n1'
CONFIG = PROJECT / 'composite_rules/configs/protection_analysis.yaml'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load chython protecting-group rules

The loader first tries the documented `chython.algorithms.groups._protective.rules` source. In the current SynPlanner environment, the same curated definitions are available through `chython.reactor.deprotection`, with the handoff copy used as a fallback.

In [10]:
from route_analysis.protection import load_chython_protection_rules

protection_rules = load_chython_protection_rules()
len(protection_rules), list(protection_rules)[:10]

(98,
 ['hydroxyl_thiocarbamate',
  'hydroxyl_fmoc',
  'hydroxyl_troc',
  'hydroxyl_teoc',
  'hydroxyl_alloc',
  'hydroxyl_tms',
  'hydroxyl_tes',
  'hydroxyl_tbs',
  'hydroxyl_tips',
  'hydroxyl_tbdps'])

## Inspect one normalized route

`normalize_route_tree` keeps the PaRoutes tree shape but makes sure reaction nodes expose mapped reaction SMILES in `node['smiles']`.

In [2]:
from route_analysis.composite_rules.extract import normalize_route_tree
from synplan.utils.visualisation import get_route_svg_from_json
from chython import depict_settings


depict_settings(aam=True)

with ROUTES_JSON.open() as f:
    routes = json.load(f)

route_id = 1
route = normalize_route_tree(routes[route_id])
display(SVG(get_route_svg_from_json({route_id: route}, route_id)))

## Analyze that route in memory

The event table tells whether a protecting group was introduced inside the route, came from stock, was ambiguous, or failed tracing.

In [ ]:
from route_analysis.protection.analysis import ProtectionAnalysisConfig, analyze_route_protection

config = ProtectionAnalysisConfig.from_yaml(CONFIG)
config.collect_interval_rules = False
events, interval_rules, route_index = analyze_route_protection(
    route,
    route_id,
    protection_rules,
    config=config,
)
pd.DataFrame([e.__dict__ fxor e in events])[[
    'event_id', 'pg_type', 'source_type', 'trace_status',
    'multicenter_status', 'lifetime_steps', 'protected_atom_ids'
]]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (647965189.py, line 11)

## Score interpretation

For one protecting-group event, `lifetime_steps` is the number of intervening reactions traced between the protection source and deprotection. The interval-rule table uses the same contiguous reaction-center-sharing rule as composite-rule extraction, then groups interval rules by QueryCGR-derived identity rather than raw SMARTS text.

Pool-level popularity is counted as:

`pg_popularity(pg) = number of detected deprotection/protection intervals with pg_type == pg`

`family_popularity(pg, family) = number of interval composite-rule observations linking pg_type to that QueryCGR family`

In [50]:
group_summary = pd.read_csv(OUTPUT_DIR / 'n1_protection_group_summary.tsv', sep='\t')
rule_families = pd.read_csv(OUTPUT_DIR / 'n1_protection_rule_families.tsv', sep='\t')
display(group_summary.head(15))
display(rule_families.head(15))

,pg_type,popularity,route_count,target_count,introduced_count,stock_count,ambiguous_count,failed_count,introduction_fraction,stock_fraction,...,max_lifetime_steps,n_unique_composite_rules,n_unique_composite_rule_families,top_composite_rule_families,top_first_reactions_after_source,top_last_reactions_before_deprotection,top_protected_functional_groups,n_multicenter_deprotection_events,n_selective_deprotections,selective_deprotection_fraction
0,amine_boc,836,814,814,123,709,4,0,0.147129,0.848086,...,8,228,218,"interval_dd7b340328c7f37c,interval_5a787b6a0cd...",[CH3:19][C:20]([CH3:21])([CH3:22])[O:23][C:24]...,[CH3:19][C:20]([CH3:21])([CH3:22])[O:23][C:24]...,amine,83,2,0.002392
1,carboxyl_methyl,748,727,727,92,656,0,0,0.122995,0.877005,...,7,195,190,"interval_6bdc6c5d8d5dccd8,interval_9b3c9795b87...",[c:9]1([C:7](=[O:8])[O:27][CH3:28])[c:10]([Cl:...,[c:24]12[n:20]([c:10]([Cl:29])[c:9]([C:7](=[O:...,carboxylic_acid,44,29,0.038770
2,carboxyl_ethyl,688,675,675,29,659,0,0,0.042151,0.957849,...,7,171,162,"interval_a92cc62359c36d54,interval_85ea8c1c358...",[Cl:23][c:22]1[cH:24][cH:25][c:26]2[c:20]([cH:...,[Cl:23][c:22]1[cH:24][cH:25][c:26]2[N:5]([CH2:...,carboxylic_acid,30,23,0.033430
3,hydroxyl_methyl,613,524,524,47,566,0,0,0.076672,0.923328,...,7,191,190,"interval_d2660f578c7f5c7b,interval_da516473f44...",[cH:24]1[c:18]([I:53])[c:19]([c:20]([n:22][cH:...,[CH3:28][C:29]1([CH3:30])[O:31][B:32]([O:33][C...,alcohol,270,153,0.249592
4,hydroxyl_benzyl,273,261,261,36,237,0,0,0.131868,0.868132,...,7,106,102,"crf_000051,interval_c8a3898d37de6450,interval_...",[F:75][C:74]([F:76])([F:77])[S:71](=[O:72])(=[...,[CH3:34][C:33]([CH3:35])([CH3:36])[O:32][C:30]...,alcohol,22,14,0.051282
5,amine_benzyl,176,164,164,29,129,18,0,0.164773,0.732955,...,5,53,51,"interval_467213bca4232b62,interval_60b09694a42...",[Cl:34][S:33]([Cl:31])=[O:32].[cH:17]1[cH:18][...,[CH3:1][C:2]([CH3:3])([CH3:4])[O:5][C:6](=[O:7...,amine,44,6,0.034091
6,hydroxyl_tbu,150,148,148,7,143,0,0,0.046667,0.953333,...,6,46,45,"interval_0683cffa92a98efc,interval_526b1dd20ee...",[CH3:33][C:34]([CH3:35])([CH3:36])[O:15][C:13]...,[CH3:33][C:34]([CH3:35])([CH3:36])[O:15][C:13]...,alcohol,6,4,0.026667
7,amine_cbz,148,148,148,19,129,0,0,0.128378,0.871622,...,6,55,51,"interval_e0d66fcd1d16eb08,interval_66d2a90a69e...",[cH:24]1[cH:23][c:22]([cH:27][cH:26][cH:25]1)[...,[CH3:15][C:14]([CH3:16])([CH3:17])[O:13][C:11]...,amine,8,0,0.000000
8,hydroxyl_tbs,124,124,124,35,89,0,0,0.282258,0.717742,...,6,30,29,"interval_17e106244cb22cf3,interval_e6524123ad8...",[CH3:43][C:44]([CH3:45])([CH3:46])[Si:47]([CH3...,[CH3:43][C:44]([CH3:45])([CH3:46])[Si:47]([CH3...,alcohol,3,0,0.000000
9,hydroxyl_acyl,117,103,103,24,91,2,0,0.205128,0.777778,...,4,7,7,"interval_3631f987a041d4cd,interval_0449664c22a...",[C:39](=[O:40])([O:41][C@@H:5]1[O:4][CH2:3][C@...,[C@@H:20]1([C@H:18]([C@H:5]([n:6]2[cH:7][n:9][...,alcohol,29,18,0.153846


,pg_type,composite_rule_family_id,family_popularity,route_count,target_count,scaffold_count,representative_composite_rule_smarts,representative_querycgr,representative_querycgr_hash,family_size_rules,median_pairwise_similarity,min_pairwise_similarity,max_pairwise_similarity,example_route_ids,example_event_ids,example_target_smiles,interpretation_label
0,amine_boc,interval_dd7b340328c7f37c,5,5,5,NaN,[c;D3:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[C;D2:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""()"", ""()"",...",dd7b340328c7f37c,1,NaN,NaN,NaN,"344,469,1943,3487,4424","1943:pg2,344:pg1,3487:pg1,4424:pg1,469:pg2",COCCN(C)c1cc2c(cc1Cl)NC(=O)CC(c1ccnc(-c3cc(C)n...,amine_boc->interval_dd7b340328c7f37c
1,carboxyl_methyl,interval_6bdc6c5d8d5dccd8,5,5,5,NaN,[c;D3:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(3,)"", ""(3...",6bdc6c5d8d5dccd8,2,NaN,NaN,NaN,"523,852,3144,3508,5077","3144:pg1,3508:pg1,5077:pg1,523:pg1,852:pg1",CC1(C)Cc2cc(C(=O)O)ccc2NC1c1ccc(NC(=O)c2ccc(F)...,carboxyl_methyl->interval_6bdc6c5d8d5dccd8
2,amine_boc,interval_5a787b6a0cdc6606,4,4,4,NaN,[c;D3:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(3,)"", ""(3...",5a787b6a0cdc6606,3,NaN,NaN,NaN,"9,2082,2784,3082","2082:pg1,2784:pg3,3082:pg1,9:pg2",COc1ccnc(-c2cnc(N)c(C(=O)Nc3ncccc3N3CCC(C)(N)C...,amine_boc->interval_5a787b6a0cdc6606
3,amine_boc,interval_e46d3c97c94878a0,4,4,4,NaN,[C;D3:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(3,)"", ""(3...",e46d3c97c94878a0,2,NaN,NaN,NaN,"37,1127,2639,4206","1127:pg3,2639:pg2,37:pg2,4206:pg3","CC1CCC(c2nc3cc(Cl)ccc3o2)CN1,CO[C@H]1CN[C@@H](...",amine_boc->interval_e46d3c97c94878a0
4,carbonyl_dioxolane,interval_6f86094f36d99ad2,4,4,4,NaN,[C;D2:1]-[C;D2:2]-[C;D3:3](-[c;D3:4])-[C;D2:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(2,)"", ""(2...",6f86094f36d99ad2,3,NaN,NaN,NaN,"319,774,3669,5685","319:pg1,3669:pg1,5685:pg1,774:pg1",CC(C)[Si](OC1=CCC(c2cc(F)c(F)cc2F)CC1)(C(C)C)C...,carbonyl_dioxolane->interval_6f86094f36d99ad2
5,carboxyl_ethyl,interval_85ea8c1c3580348e,4,3,3,NaN,[C;D2:1]-[C;D3:2](=[O;D1:3])-[N;D2:4]-[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(2,)"", ""(2...",85ea8c1c3580348e,2,NaN,NaN,NaN,"1252,2009,9917","1252:pg1,2009:pg1,9917:pg1,9917:pg2",CC(C)(C)CC(=O)Nc1ccc2c(c1)cc(C(=O)Nc1ccc(NC(=O...,carboxyl_ethyl->interval_85ea8c1c3580348e
6,carboxyl_ethyl,interval_a92cc62359c36d54,4,4,4,NaN,[C;D3:1]-[c;D3:2]:1:[c;D2:3]:[s;D2:4]:[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(2,)"", ""(2...",a92cc62359c36d54,3,NaN,NaN,NaN,"61,124,915,4627","124:pg1,4627:pg1,61:pg1,915:pg1",CC(C)CON=C(C(=O)NC1C(=O)N2C(C(=O)O)=CCS[C@H]12...,carboxyl_ethyl->interval_a92cc62359c36d54
7,amine_benzyl,interval_467213bca4232b62,3,3,3,NaN,[O;D1:1]=[C;D3:2]-1-[N;D3:3](-[c;D3:4])-[C;D2:...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""()"", ""()"",...",467213bca4232b62,3,NaN,NaN,NaN,"645,2312,6339","2312:pg1,6339:pg1,645:pg1",CCCn1cccc(N2CCC3(CC2)CCN(c2ccc(OC(F)(F)F)cc2)C...,amine_benzyl->interval_467213bca4232b62
8,amine_boc,interval_5ebc469c033b7295,3,3,3,NaN,[C;D2:1]-[N;D1:2]>>[C;D2:1]-[N;D2:2]-[C;D2:7]-...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""()"", ""()"",...",5ebc469c033b7295,1,NaN,NaN,NaN,"21,858,3101","21:pg3,3101:pg2,858:pg2",COc1c(F)cccc1C(=O)N1C[C@@H](C)CC[C@H]1CNc1ccc(...,amine_boc->interval_5ebc469c033b7295
9,amine_cbz,interval_e0d66fcd1d16eb08,3,3,3,NaN,[C;D3:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[c;D3:5]...,"[[[[6, ""0"", ""0"", ""False"", ""False"", ""(3,)"", ""(3...",e0d66fcd1d16eb08,2,NaN,NaN,NaN,"934,2513,7902","2513:pg2,7902:pg2,934:pg3",CC(C)C[C@H](NC(=O)c1cc2ccccc2o1)C(=O)NC1CCCN(C...,amine_cbz->interval_e0d66fcd1d16eb08


## Visualize a route with a detected event

The route JSON remains compatible with `get_route_svg_from_json`, so a detected event can be inspected by route id from the event table.

In [51]:
events_tsv = OUTPUT_DIR / 'n1_protection_events.tsv'
summary_json = OUTPUT_DIR / 'n1_protection_summary.json'
events_df = pd.read_csv(events_tsv, sep='\t')
events_df.head(10)

,event_id,route_id,target_smiles,pg_type,protected_functional_group,source_type,trace_status,confidence,deprotection_node_id,deprotection_reaction_smiles,...,depth_deprotection_from_target,depth_source_from_target,lifetime_steps,n_intervening_steps,multicenter_status,n_other_reaction_centers,n_matching_pg_sites_before_deprotection,n_sites_deprotected,selective_deprotection,failure_reason
0,0:pg1,0,CC(C)(C)[C@@H](CS(C)(=O)=O)Nc1nc(-c2c[nH]c3ncc...,amine_tosyl,amine,stock,stock,exact,r0,[CH3:1][C:2]([CH3:3])([CH3:4])[C@H:5]([NH:11][...,...,1,2.0,2,2,single_center_deprotection,0,1,1,False,NaN
1,1:pg1,1,COc1ccc(F)c(-c2ccc(COc3cccc(C(CC(=O)O)C4CC4)c3...,hydroxyl_tbdps,alcohol,introduced,introduced,exact,r4,[cH:57]1[cH:56][c:55]([cH:60][cH:59][cH:58]1)[...,...,3,5.0,2,2,single_center_deprotection,0,1,1,False,NaN
2,1:pg2,1,COc1ccc(F)c(-c2ccc(COc3cccc(C(CC(=O)O)C4CC4)c3...,carboxyl_methyl,carboxylic_acid,introduced,introduced,high,r0,[CH3:10][C:9]([CH3:11])([CH3:12])[CH2:8][c:7]1...,...,1,3.0,2,2,single_center_deprotection,0,1,1,False,NaN
3,1:pg3,1,COc1ccc(F)c(-c2ccc(COc3cccc(C(CC(=O)O)C4CC4)c3...,hydroxyl_methyl,alcohol,stock,stock,exact,r0,[CH3:10][C:9]([CH3:11])([CH3:12])[CH2:8][c:7]1...,...,1,7.0,7,7,deprotection_plus_other,2,1,1,False,NaN
4,2:pg1,2,CCCCCCOc1ccc(C(=O)Nc2ccccc2Oc2cccc(OC(C)c3nnn[...,hydroxyl_methyl,alcohol,stock,stock,exact,r7,[cH:19]1[c:18]([cH:31][cH:30][cH:29][c:20]1[O:...,...,8,9.0,2,2,single_center_deprotection,0,1,1,False,NaN
5,2:pg2,2,CCCCCCOc1ccc(C(=O)Nc2ccccc2Oc2cccc(OC(C)c3nnn[...,carboxyl_ethyl,carboxylic_acid,stock,stock,high,r5,[O:41]=[C:24]([O:42][CH2:44][CH3:43])[CH:22]([...,...,6,7.0,2,2,single_center_deprotection,0,1,1,False,NaN
6,4:pg1,4,NC[C@H]1CC[C@@H](C(O)CCc2ccccc2)CC1,carboxyl_butyl,carboxylic_acid,stock,stock,high,r4,[CH3:19][C:20]([CH3:21])([CH3:22])[O:23][C:24]...,...,5,10.0,6,6,single_center_deprotection,0,1,1,False,NaN
7,4:pg2,4,NC[C@H]1CC[C@@H](C(O)CCc2ccccc2)CC1,amine_boc,amine,introduced,introduced,exact,r0,[CH3:19][C:20]([CH3:21])([CH3:22])[O:23][C:24]...,...,1,6.0,5,5,single_center_deprotection,0,1,1,False,NaN
8,5:pg1,5,COc1cc(Cl)c(C2CCC2)cc1NCC(=O)N1CCN(C2CN(C(=O)O...,carboxyl_ethyl,carboxylic_acid,stock,stock,high,r2,[CH2:39]([O:37][C:7]([CH2:6][NH:5][c:4]1[c:3](...,...,3,4.0,2,2,single_center_deprotection,0,1,1,False,NaN
9,6:pg1,6,CCN(c1cccc2c1CCCCC(C)c1cc(C)[nH]c(=O)c1CNC2=O)...,carboxyl_methyl,carboxylic_acid,introduced,introduced,high,r5,[CH2:29]1[CH2:30][O:31][CH2:32][CH2:33][CH:28]...,...,5,9.0,4,4,single_center_deprotection,0,1,1,False,NaN


In [52]:
from chython import smiles, depict_settings

depict_settings(monochrome=False)

In [53]:
x = 344
if not events_df.empty:
    example_route_id = int(events_df.iloc[x]['route_id'])
    example_route = normalize_route_tree(routes[example_route_id])
    print(events_df.iloc[x][['event_id', 'pg_type', 'source_type', 'trace_status', 'lifetime_steps']])
    display(SVG(get_route_svg_from_json({example_route_id: example_route}, example_route_id, labeled=True)))
    display(smiles(events_df.iloc[x]['deprotection_reaction_smiles']))

event_id               382:pg2
pg_type           hydroxyl_tbs
source_type         introduced
trace_status        introduced
lifetime_steps               3
Name: 344, dtype: object


In [70]:
multi_df = events_df[events_df['multicenter_status'] == 'deprotection_plus_other']
multi_df[:20]

,event_id,route_id,target_smiles,pg_type,protected_functional_group,source_type,trace_status,confidence,deprotection_node_id,deprotection_reaction_smiles,...,depth_deprotection_from_target,depth_source_from_target,lifetime_steps,n_intervening_steps,multicenter_status,n_other_reaction_centers,n_matching_pg_sites_before_deprotection,n_sites_deprotected,selective_deprotection,failure_reason
2,1:pg2,1,COc1ccc(F)c(-c2ccc(COc3cccc(C(CC(=O)O)C4CC4)c3...,hydroxyl_methyl,alcohol,introduced,introduced,exact,r0,C[O:26][C:24]([CH2:23][CH:22]([c:21]1[cH:20][c...,...,1,3.0,1,1,deprotection_plus_other,2,2,1,True,NaN
7,6:pg2,6,CCN(c1cccc2c1CCCCC(C)c1cc(C)[nH]c(=O)c1CNC2=O)...,hydroxyl_methyl,alcohol,introduced,introduced,exact,r1,C[O:23][c:22]1[c:17]2[c:16]([cH:27][c:25]([CH3...,...,2,4.0,1,1,deprotection_plus_other,1,1,1,False,NaN
12,10:pg1,10,CC(=O)N1CCC(c2csc(Nc3ncc(SCCCO)cc3Oc3ccccc3)n2...,hydroxyl_methyl,alcohol,introduced,introduced,high,r0,C[O:21][C:20](=O)[CH2:19][CH2:18][S:17][c:16]1...,...,1,2.0,0,0,deprotection_plus_other,2,1,1,False,NaN
17,14:pg1,14,O=C(O)CCCNC(=O)c1ncc2c(cc(-c3ccccc3)c(=O)n2CC2...,amine_tosyl,amine,stock,stock,high,r3,COC(=O)C[N:6](S(=O)(=O)[c:5]1ccc(C)c[cH:10]1)[...,...,4,5.0,1,1,deprotection_plus_other,7,1,1,False,NaN
19,17:pg1,17,Cc1cc(C)cc(-c2c(OCC[C@@H]3CCCCN3)c3cc(NC(=O)NC...,hydroxyl_methyl,alcohol,stock,stock,high,r4,C[O:16][C:15](=O)[CH2:14][C@H:13]1[N:8]([C:6](...,...,5,5.0,0,0,deprotection_plus_other,2,1,1,False,NaN
21,18:pg1,18,CN1CCC(Oc2ccc3[nH]nc(S(=O)(=O)c4cccc5ccccc45)c...,amine_benzyl,amine,introduced,introduced,high,r4,Cl[C:2](=[O:1])[O:3][CH2:4][c:5]1[cH:6][cH:7][...,...,5,6.0,0,0,deprotection_plus_other,2,1,1,False,NaN
24,21:pg1,21,Cc1cccc(-c2sc(C)nc2C(=O)N2C[C@@H]3CC(C)C[C@@H]...,amine_benzyl,amine,stock,stock,high,r8,CC(C)(C)OC(=O)O[C:9](=[O:10])[O:11][C:12]([CH3...,...,9,9.0,0,0,deprotection_plus_other,2,1,1,False,NaN
39,38:pg1,38,CCOC(=O)CCc1ccc(-c2ccc(CN3CCN(C)CC3)cc2)c(OCCC...,hydroxyl_methyl,alcohol,stock,stock,exact,r5,C[O:9][c:8]1[c:7]([O:6][CH2:5][CH2:4][CH2:3][O...,...,6,7.0,1,1,deprotection_plus_other,2,2,1,True,NaN
50,53:pg1,53,O=C1CC(c2cccc(-n3ccnc3)c2)=Nc2ccc(C#Cc3ccc(F)c...,amine_boc,amine,introduced,introduced,high,r0,CC(C)(C)OC(=O)[NH:16][c:17]1[c:18]([NH:32][C:2...,...,1,2.0,0,0,deprotection_plus_other,2,1,1,False,NaN
52,54:pg2,54,COc1ccc(-c2ncc(F)c(N(C)CCCOc3ccc4c(ccn4CC(=O)O...,hydroxyl_methyl,alcohol,stock,stock,exact,r0,C[O:31][C:29]([CH2:28][n:27]1[c:22]2[cH:21][cH...,...,1,3.0,2,2,deprotection_plus_other,2,2,1,True,NaN


In [ ]:
from chython import smarts, smiles
import pandas as pd
from synplan.utils.visualisation import get_route_svg_from_json
from route_analysis.alchemical_rules.unwrap_alchemical import unwrap_alchemical_rule
from IPython.display import display, SVG
from route_analysis.composite_rules.unwrap import split_composite_rule, unwrap_composite_rule

In [57]:
label= 'n1'
import json

routes_path = f"../data/{label}-routes.json"

with open(routes_path) as f:
    routes = json.load(f)

# PaRoutes file is a list, while get_route_svg_from_json expects route_id lookup.
routes_json = {i: route for i, route in enumerate(routes)}


In [64]:
route_id = 9897
# route_id = 9895 # not clear
# route_id = 9872
# route_id = 9859
route_id = 461
svg = get_route_svg_from_json(routes_json, route_id, labeled=True)
display(SVG(svg))

In [61]:
route_id = 4424
svg = get_route_svg_from_json(routes_json, route_id, labeled=True)
display(SVG(svg))

In [14]:
from chython import smarts
# smarts('[C;D3,D4;z1;x2:1](-;!@[O;D2;x0][C;D1])-;!@[O;D2;x0][C;D1]')
smarts('[N;D2,D3:1]-;!@[C;z2;x3](=O)[C;D1]')

In [62]:
from route_analysis.composite_rules.unwrap import split_composite_rule

rules = split_composite_rule('[c;D3:1]-[N;D2:2]-[C;D3:3](=[O;D1:4])-[C;D2:5]>>[c;D3:1]-[N;D1:2].[C;D3:3](=[O;D1:4])(-[C;D2:5])-[O;D2:10]-[C;D4:7](-[C:6])(-[C:8])-[C:9]$[c;D3:1]-[N;D1:2]>>[c;D3:1]-[N;D3+:2](=[O;D1:3])-[O;D1-:4]')
for rule in rules:
    display(smarts(rule))

#### Absent protecting groups in PaRoutes

In [75]:
g = pd.read_csv('/Users/almazgil/Desktop/projects/Retro-BLEU/composite_rules/protection_out/n1/n1_protection_group_summary.tsv', sep='\t')
found_pg_rules = set(g['pg_type'])
print(len(found_pg_rules))

45


In [76]:
all_pg_rules = set(protection_rules.keys())

In [77]:
absent_pg = all_pg_rules - found_pg_rules
print(len(absent_pg))
absent_pg

53


{'amine_bhoc',
 'amine_chloro_cbz',
 'amine_chloro_tritil',
 'amine_dde_enamine',
 'amine_dde_imine',
 'amine_ivdde_enamine',
 'amine_ivdde_imine',
 'amine_mtr',
 'amine_mtt',
 'amine_nosyl',
 'amine_pbf',
 'carbonyl_dimethylsulfide',
 'carbonyl_dithiane',
 'carbonyl_dithiolane',
 'carboxyl_trioxabicyclooctane',
 'diol_12_benzylidene',
 'diol_12_cyclohexanone',
 'diol_12_cyclopentanone',
 'diol_12_diacetal',
 'diol_12_formalin',
 'diol_13_benzylidene',
 'diol_13_cyclohexanone',
 'diol_13_cyclopentanone',
 'diol_13_diacetal',
 'diol_13_formalin',
 'hydroxyl_alloc',
 'hydroxyl_bom',
 'hydroxyl_chloro_tritil',
 'hydroxyl_dimethoxybenzyl',
 'hydroxyl_dimetoxy_tritil',
 'hydroxyl_dmab_enamine',
 'hydroxyl_dmab_imine',
 'hydroxyl_ee',
 'hydroxyl_fmoc',
 'hydroxyl_methoxy_benzoate',
 'hydroxyl_mmt',
 'hydroxyl_mop',
 'hydroxyl_mpe',
 'hydroxyl_naphthyl',
 'hydroxyl_o_nitrobenzyl',
 'hydroxyl_teoc',
 'hydroxyl_tfa',
 'hydroxyl_thiocarbamate',
 'hydroxyl_trifluoroethyl',
 'hydroxyl_troc',
 'thi